# Combinações das bandas `RGB`, exemplo para o `Landsat-8` e `Sentinel-2`

1)  [Landsat-8](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2):
    *   `SR_B4:` 0.636-0.673 μm - Vermelho (R)
    *   `SR_B3:` 0.533-0.590 μm - Verde (G)
    *   `SR_B2:` 0.452-0.512 μm - Azul (B)


2) [Sentinel-2](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED):
    *   `B4:` 0.664 μm (S2A) / 0.665 μm (S2B) - Vermelho (R)
    *   `B3:` 0.560 μm (S2A) / 0.559 μm (S2B) - Verde (G)
    *   `B2:` 0.496 μm (S2A) / 0.492 μm (S2B) - Azul (B)

# Inicializando o GEE

In [ ]:
# importando o GEE e Geemap
import ee
import geemap

# inicializando GEE
geemap.ee_initialize(project='ee-enrique')

# Carregando os dados e aplicando o fator de escala

In [ ]:
#==============================================================================================================#
#                                       SHAPEFILE DO ESTADO DO RS
#==============================================================================================================#
rs = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.eq('ADM1_NAME', 'Rio Grande Do Sul'))

#==============================================================================================================#
#                                             LANDSAT-8
#==============================================================================================================#
# aplica fator de escala
def fator_escala(img):

    # Aplica fatores de escala
    opticas = img.select('SR_B.*').multiply(2.75e-05).add(-0.2)
    termais = img.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Retorna imagem com bandas escaladas
    return (img.addBands(opticas, None, True)
               .addBands(termais, None, True)
               .copyProperties(img, ['system:time_start'])
               .set('date', img.date().format('YYYY-MM-dd')))

# dados: https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2
landsat8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
             .filterBounds(rs) \
             .filterDate('2024-01-01','2025-01-01') \
             .filter(ee.Filter.lt('CLOUD_COVER', 5)) \
             .sort('CLOUD_COVER', True) \
             .map(fator_escala)

#==============================================================================================================#
#                                               SENTINEL-2
#==============================================================================================================#
# dados: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED
sentinel2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
              .filterBounds(rs) \
              .filterDate('2024-01-01','2025-01-01') \
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
              .sort('CLOUDY_PIXEL_PERCENTAGE', True) \
              .map(lambda img: img.multiply(0.0001).copyProperties(img, img.propertyNames()))

In [ ]:
# dados Landsat-8
landsat8

In [ ]:
# dados Sentinel-2
sentinel2

# Plotando mapa

In [ ]:
# parâmetros de visualização
param_vis_landsat_8 = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0.03, 'max': 0.19}
param_vis_sentinel_2 = {'bands': ['B4','B3','B2'], 'min': 0.03, 'max': 0.19}

# monta painel com 2 mapas
geemap.linked_maps(rows = 1,
                   cols = 2,
                   height = "400px",
                   center = [-29.94, -51.32],
                   zoom = 11,
                   ee_objects = [landsat8.mean().clip(rs), sentinel2.mean().clip(rs)],
                   vis_params = [param_vis_landsat_8, param_vis_sentinel_2],
                   labels = ['RGB Landsat-8', 'RGB Sentinel-2'],
                   label_position = "topright")

# Para Porto Alegre

In [ ]:
# define regiao
porto_alegre = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Porto Alegre'))

# criando um mapa interativo
Map = geemap.Map()

# centra o mapa na região
Map.centerObject(porto_alegre, zoom=10)

# adicionando basemap
Map.add_basemap('Esri.WorldImagery')

# contorno de Porto Alegre
style = {'color': 'yellow', 'fillColor': '00000000'}
Map.addLayer(porto_alegre.style(**style), {}, 'itajuba')

# exibe na tela
Map

In [ ]:
#==============================================================================================================#
#                                       SHAPEFILE DO MUNICÍPIO DE PORTO ALEGRE
#==============================================================================================================#
porto_alegre = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Porto Alegre'))

#==============================================================================================================#
#                                             LANDSAT-8
#==============================================================================================================#
# aplica fator de escala
def fator_escala(img):

    # Aplica fatores de escala
    opticas = img.select('SR_B.*').multiply(2.75e-05).add(-0.2)
    termais = img.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Retorna imagem com bandas escaladas
    return (img.addBands(opticas, None, True)
               .addBands(termais, None, True)
               .copyProperties(img, ['system:time_start'])
               .set('date', img.date().format('YYYY-MM-dd')))

# dados: https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2
landsat8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
             .filterBounds(porto_alegre) \
             .filterDate('2024-01-01','2025-01-01') \
             .filter(ee.Filter.lt('CLOUD_COVER', 1)) \
             .sort('CLOUD_COVER', True) \
             .map(fator_escala)

#==============================================================================================================#
#                                               SENTINEL-2
#==============================================================================================================#
# dados: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED
sentinel2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
              .filterBounds(porto_alegre) \
              .filterDate('2024-01-01','2025-01-01') \
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 1)) \
              .sort('CLOUDY_PIXEL_PERCENTAGE', True) \
              .map(lambda img: img.multiply(0.0001).copyProperties(img, img.propertyNames()))

#==============================================================================================================#
#                                               MAPA INTERATIVO
#==============================================================================================================#
# parâmetros de visualização
param_vis_landsat_8 = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0.03, 'max': 0.19}
param_vis_sentinel_2 = {'bands': ['B4','B3','B2'], 'min': 0.03, 'max': 0.19}

# monta painel com 2 mapas
geemap.linked_maps(rows = 1,
                   cols = 2,
                   height = "400px",
                   center = [-30.18, -51.22],
                   zoom = 14,
                   ee_objects = [landsat8.mean().clip(rs), sentinel2.mean().clip(rs)],
                   vis_params = [param_vis_landsat_8, param_vis_sentinel_2],
                   labels = ['RGB Landsat-8', 'RGB Sentinel-2'],
                   label_position = "topright")